<div style="border:2px solid #B3241C; background:#FDECEA; border-radius:8px; padding:16px 20px;">
  <div style="color:#B3241C; font-size:26px; font-weight:800; line-height:1.25; margin-bottom:8px;">
    STOP &mdash; do this first
  </div>
  <div style="font-size:19px; color:#1F2328; line-height:1.5;">
    <b>File &rarr; Save a copy in Drive</b>, then work in <b>your</b> copy.<br>
    If you skip this step, nothing you type here will be saved.
  </div>
</div>

# Week 3, Session 2 — Holding Data Back

Ten minutes. Nothing here is hard, and that is on purpose.

You have just seen why a model that fits its data perfectly can still be worthless.
The fix is to **not give the model all of the data**. Hide some, train on the rest,
and test on what you hid.

This notebook shows you what that split actually does. In Week 4 you will use it for real.

**Run every cell in order.** If anything looks broken, use `Runtime -> Restart and run all` —
that fixes most Colab strangeness.

## Setup

Run this cell. You do not need to change anything in it. It is OK that you don't understand the code. Just know that a DataFrame of 400 customers of a telecom company is created with this cell

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option('display.precision', 3)
random_seed = 42

# A small made-up customer table, so this notebook never depends on a file.
# 400 customers. Customers who call support more often are likelier to churn.
rng = np.random.default_rng(random_seed)
n = 400
months = rng.integers(1, 72, n)
spend  = rng.normal(85, 25, n).round(2).clip(15)
calls  = rng.poisson(1.2, n)

risk = 1 / (1 + np.exp(-(-1.55 + 0.45 * calls - 0.02 * months)))

customers = pd.DataFrame({
    'customer_id':   range(1001, 1001 + n),
    'months_active': months,
    'monthly_spend': spend,
    'support_calls': calls,
    'plan':          rng.choice(['Basic', 'Plus', 'Premium'], n, p=[.5, .35, .15]),
    'churned':       (rng.random(n) < risk).astype(int),
})

print('Rows, columns:', customers.shape)
customers.head()

Rows, columns: (400, 6)


,customer_id,months_active,monthly_spend,support_calls,plan,churned
0,1001,7,101.08,2,Plus,0
1,1002,55,75.13,3,Basic,0
2,1003,47,84.87,4,Plus,1
3,1004,32,80.91,1,Basic,0
4,1005,31,93.44,0,Premium,0


## 1. What are we splitting?

Before any split, look at what you have. Two questions you should now ask automatically:

- **What is the unit of analysis?** One row = one what?
- **What is the target variable?**

In [ ]:
# X = the features (everything we predict FROM)
# y = the target    (the thing we predict)
X = customers.drop(columns=['churned', 'customer_id'])
y = customers['churned']

print('X shape:', X.shape)
print('y shape:', y.shape)
print()
print('How common is churn in the full dataset?')
print(y.value_counts(normalize=True).round(3))

X shape: (400, 4)
y shape: (400,)

How common is churn in the full dataset?
churned
0    0.812
1    0.188
Name: proportion, dtype: float64


> **Why did we drop `customer_id`?** Think about it before you read on. I assume you know why we drop `churned` but be sure to ask if you don't. This is important.
>
> It is a unique identifier. It carries no information about whether someone churns —
> every customer has a different one. Feeding it to a model adds noise and nothing else.

## 2. The split

`train_test_split()` shuffles the rows and cuts them into two groups: training and test

**Fill in the two blanks.** Set the test set to 25% of the data, and set `random_state`
to the `random_seed` defined at the top.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,        # <-- 25% of the data
    random_state=19      # <-- use random_seed
)

print('Training rows:', len(X_train))
print('Test rows:    ', len(X_test))
print('Total:        ', len(X_train) + len(X_test), ' (should equal', len(X), ')')

Training rows: 300
Test rows:     100
Total:         400  (should equal 400 )


## 3. Look at what happened

Run this. The training and test sets are different rows — no customer is in both.

In [5]:
print('First 5 training rows (by index):', list(X_train.index[:5]))
print('First 5 test rows     (by index):', list(X_test.index[:5]))
print()
overlap = set(X_train.index) & set(X_test.index)
print('Customers in BOTH sets:', len(overlap))

First 5 training rows (by index): [49, 359, 86, 65, 54]
First 5 test rows     (by index): [133, 303, 181, 276, 321]

Customers in BOTH sets: 0


## 4. Why `random_state` matters

Run the next cell **twice**. The result is identical both times, because `random_state`
fixes the shuffle.

Then change `random_state` to `7` and run it again. Different rows.

This is why every assignment tells you which `random_state` to use — without it,
your numbers would never match mine.

In [6]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=random_seed)
print('First 5 test rows:', list(Xte.index[:5]))

First 5 test rows: [209, 280, 33, 210, 93]


## 5. One thing that can go wrong

Fewer than one customer in five churned. The split is random, so by bad luck the test
set can end up with a very different churn rate than the real data — and then your test
results describe a population that does not exist.

`stratify=y` tells the split to keep the same proportion of churners in both pieces.

Run this and compare the three numbers.

In [7]:
# The same split, done both ways, with the same random_state
_, _, _, y_te_plain = train_test_split(
    X, y, test_size=0.25, random_state=164)

_, _, _, y_te_strat = train_test_split(
    X, y, test_size=0.25, random_state=164, stratify=y)

print('Churn rate in the full data:', round(y.mean(), 3))
print('Test set WITHOUT stratify: ', round(y_te_plain.mean(), 3))
print('Test set WITH stratify:    ', round(y_te_strat.mean(), 3))

Churn rate in the full data: 0.188
Test set WITHOUT stratify:  0.29
Test set WITH stratify:     0.19


Without `stratify`, this test set has a churn rate far above the real one. Any model
you evaluated on it would be judged against a world that is not the one you operate in.

**Use `stratify=y` whenever your target is a category.** You will do this in Week 5.
For predicting a number, you do not need it.

## Before you close this

Answer in the cell below, in one sentence each. No code.

1. Why can't we judge a model using the same rows we trained it on?
2. Your test set turns out to contain zero churners. Why is that a problem?

In [ ]:
# Your answers here (this is a code cell -- just put a # in front of each line)
#
# 1.
#
# 2.

### Swapping in a real file later

When you need your own CSV instead of the made-up table above, replace the setup cell with:

```python
customers = pd.read_csv('your_file.csv')
```

Everything below it works unchanged.